---
---
# Part II — Medicaid Expansion (ACA, 2014): A Second DiD Case Study
**ECON 4370 — Applied Data Tools for Economics**  
**Dr. Fidel González — Spring 2026**

---

In 2010, the **Affordable Care Act (ACA)** directed all states to expand Medicaid
coverage to adults with incomes up to 138% of the federal poverty line, effective January 2014.

In **NFIB v. Sebelius (2012)**, the Supreme Court upheld the ACA but struck down
the expansion mandate — making it **optional**. States could choose whether to expand.

As of January 1, 2014:
- **26 states (+ D.C.) expanded** → treated group
- **24 states did not expand** → control group
- **Outcome of interest:** share of non-elderly adults without health insurance (%)

This creates a natural experiment almost identical to Card & Krueger:
a policy shock that hit some states but not others on a known date.

**The interesting wrinkle:** expansion states already had *lower* baseline uninsured
rates than non-expansion states. That is fine for DiD — we only need parallel *trends*,
not equal *levels*. But it raises a real question: were expansion states already on a
steeper downward path before 2014, independent of the policy? This is the central
parallel-trends debate in the Medicaid literature — and we will simulate both versions.

---

**What this section adds beyond Card & Krueger:**
- 4 time periods (2 pre, 2 post) instead of 2 → allows a proper **event study**
- State-level panel instead of firm-level → different unit of analysis
- Parallel trends assumption is genuinely *contested* → richer discussion
- Outcome goes *down* with treatment → tests sign intuition

**Calibration (you know the truth):**
- True treatment effect: **−4.5 percentage points** (based on published estimates)
- Common annual trend: **−0.8 pp/year** (uninsurance was declining for everyone pre-ACA)
- Noise: realistic state-level variation

> **Mindset reminder:** Write your prior *before* running each cell.
> Knowing the true effect is −4.5 pp does not mean your estimator will recover it exactly.
> Understanding the gap is half the lesson.


---
## Part II — Learning Objectives

By the end of this section you can:

1. Apply the DiD framework to a **multi-period, state-level panel** dataset.
2. Construct a **2×2 DiD table** and regression using a new outcome and context.
3. Build and interpret a **4-period event study** — the standard tool for testing parallel trends.
4. Diagnose **biased estimates** when expansion states had a pre-existing trend.
5. Compare the Card & Krueger and Medicaid designs — similarities, differences, and credibility.


---
## 13) Setup Note

This section runs in the **same environment** as Part I.
All libraries (numpy, pandas, matplotlib, statsmodels) and folder paths
(ROOT, RAW_DIR, CLEAN_DIR, EXPORT_DIR) were defined in Section 1 above.

The color palette (NAVY, TEAL, GOLD, RED, GRAY) and plot style carry over as well.

If you are running this notebook standalone, execute the cell below.
If you are continuing from Part I in the same kernel, skip it.


In [ ]:
# ── Run this cell ONLY if starting fresh (not continuing from Part I) ──────
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import statsmodels.formula.api as smf
    STATSMODELS = True
except ImportError:
    STATSMODELS = False
    print("statsmodels not found — regression section will print 2x2 estimate only.")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.3f}".format)

ROOT       = Path.cwd() / "lecture_did"
RAW_DIR    = ROOT / "data_raw"
CLEAN_DIR  = ROOT / "data_clean"
EXPORT_DIR = ROOT / "exports"
for d in [RAW_DIR, CLEAN_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})

NAVY = "#1A2B4A"
TEAL = "#2A9D8F"
GOLD = "#E9A42E"
RED  = "#C0392B"
GRAY = "#8A9AAA"

print("Setup complete.")

---
## 14) Simulate the Data — Medicaid Expansion

We build a **state-year panel**: 45 synthetic states observed over 4 years.
This mirrors the aggregate data structure used in published Medicaid studies
(e.g., Sommers et al. 2015; Frean, Gruber & Sommers 2017).

**Design decisions to notice:**
- Each `state_id` has a random **state fixed effect** — a permanent deviation from the group mean.
  This mimics the fact that California and Montana are both expansion states but differ
  systematically. DiD absorbs this via the double-difference.
- Both groups share the same `TIME_TREND_MED`. If you only looked at the raw decline in
  expansion states, you would *overstate* the policy effect.
- The outcome **decreases** (fewer people uninsured). Make sure your sign intuition is correct
  before running the regression.

| Parameter | Value | Meaning |
|:---|---:|:---|
| `N_EXP` | 26 | Expansion states |
| `N_NON` | 19 | Non-expansion states |
| `BASELINE_EXP` | 14.5% | Uninsured rate in expansion states, 2012 |
| `BASELINE_NON` | 19.0% | Uninsured rate in non-expansion states, 2012 |
| `TIME_TREND_MED` | −0.8 pp/yr | Common annual decline (affects **both** groups) |
| `TRUE_EFFECT_MED` | **−4.5 pp** | Causal effect of Medicaid expansion |
| `NOISE_SD_MED` | 1.2 pp | State-level idiosyncratic noise |

> **Note the baseline asymmetry:** Non-expansion states start *higher* (19.0%) than
> expansion states (14.5%). This is the reverse of Card & Krueger, where the treated
> group (NJ) had a higher baseline. DiD handles both cases identically — levels can differ.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 14) Simulate the Medicaid Expansion panel
#
#     Unit:    State × Year (45 states, 4 years → 180 observations)
#     Outcome: uninsured_rate — % of non-elderly adults without insurance
#     Years:   2012, 2013 (pre-expansion) | 2014, 2015 (post-expansion)
#     Truth:   TRUE_EFFECT_MED = -4.5 pp  ← your DiD should recover this
# ─────────────────────────────────────────────────────────────────────────────
np.random.seed(99)

# ── Calibrated parameters ──────────────────────────────────────────────────
N_EXP           = 26       # expansion states (+ D.C., rounded for simplicity)
N_NON           = 19       # non-expansion states
YEARS_MED       = [2012, 2013, 2014, 2015]
TREAT_YEAR_MED  = 2014

BASELINE_EXP    = 14.5     # % uninsured, expansion group, 2012
BASELINE_NON    = 19.0     # % uninsured, non-expansion group, 2012
TIME_TREND_MED  = -0.8     # common annual pp decline (secular trend, hits both groups)
TRUE_EFFECT_MED = -4.5     # causal effect of Medicaid expansion (pp)
NOISE_SD_MED    = 1.2      # state-level noise (pp)
STATE_FE_SD     = 0.9      # state fixed-effect spread — states differ permanently

# ── Build state-year panel ─────────────────────────────────────────────────
rows_med = []
for state_id in range(N_EXP + N_NON):
    is_exp   = (state_id < N_EXP)                     # True → expansion state
    state_fe = np.random.normal(0, STATE_FE_SD)        # permanent state deviation
    baseline = BASELINE_EXP if is_exp else BASELINE_NON

    for year in YEARS_MED:
        t       = year - 2012                          # 0, 1, 2, 3
        post    = int(year >= TREAT_YEAR_MED)          # 0 for 2012/2013, 1 for 2014/2015
        effect  = TRUE_EFFECT_MED if (is_exp and post) else 0.0
        noise   = np.random.normal(0, NOISE_SD_MED)   # iid state-year noise

        unins_rate = baseline + TIME_TREND_MED * t + effect + state_fe + noise

        rows_med.append({
            "state_id"      : state_id,
            "year"          : year,
            "expansion"     : int(is_exp),
            "post"          : post,
            "year_rel"      : year - TREAT_YEAR_MED,   # -2, -1, 0, +1
            "uninsured_rate": round(unins_rate, 3),
            "group_label"   : "Expansion" if is_exp else "Non-Expansion"
        })

df_med = pd.DataFrame(rows_med)
df_med["treated_x_post"] = df_med["expansion"] * df_med["post"]

# Save raw file (mirrors Part I convention)
df_med.to_csv(RAW_DIR / "medicaid_expansion_raw.csv", index=False)

print(f"Shape: {df_med.shape}")
print(f"Groups: {df_med['group_label'].unique()}")
print(f"Years:  {sorted(df_med['year'].unique())}")
print(f"\nTrue DiD parameter = {TRUE_EFFECT_MED} pp  "
      f"(your estimator should land close to this)")
print()

# Preview group means by year
(df_med.groupby(["group_label", "year"])["uninsured_rate"]
       .mean()
       .round(2)
       .rename("mean_uninsured_rate"))

---
## 15) Explore the Medicaid Data

Before any analysis — understand the shape of the data.
Compare these summary statistics to the Card & Krueger summaries from Part I.
Notice anything structurally different?


In [ ]:
# 15.1) Structure check
print("=== Shape ===")
print(df_med.shape)

print("\n=== First 8 rows ===")
display(df_med.head(8))

print("=== Observations by group × year ===")
display(
    df_med.groupby(["group_label", "year"])
          .size()
          .rename("n_states")
          .reset_index()
)

In [ ]:
# 15.2) Summary statistics by group × year
#
# Read each row carefully:
#   - Which group has a higher baseline uninsured rate?
#   - Is the common time trend visible?
#   - Can you already "see" the treatment effect in the 2014 / 2015 rows?

summary_med = (
    df_med.groupby(["group_label", "year"])["uninsured_rate"]
          .agg(Mean="mean", SD="std", Min="min", Max="max", N="count")
          .round(3)
)
display(summary_med)

In [ ]:
# 15.3) Quick visual — raw trends before any DiD math
#
# This is an exploratory plot — NOT the formal DiD visualization.
# Just look at the raw group means over time.

gmeans_raw = df_med.groupby(["group_label", "year"])["uninsured_rate"].mean()

fig, ax = plt.subplots(figsize=(9, 5))

for label, color, marker in [("Expansion", TEAL, "o"), ("Non-Expansion", NAVY, "s")]:
    vals = [gmeans_raw[label][y] for y in YEARS_MED]
    ax.plot(YEARS_MED, vals, color=color, marker=marker,
            linewidth=2.5, markersize=8, label=label)
    for y, v in zip(YEARS_MED, vals):
        ax.text(y, v + 0.15, f"{v:.1f}%", ha="center",
                fontsize=9, color=color)

ax.axvline(x=2013.5, color=GRAY, linestyle=":", linewidth=1.5, alpha=0.8)
ax.text(2013.55, summary_med["Mean"].max() * 1.01,
        "Expansion\nstarts →", color=GRAY, fontsize=9, va="top")

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Mean Uninsured Rate (%)", fontsize=12)
ax.set_title("Raw Group Means — Medicaid Expansion Study\n"
             "(exploratory; no counterfactual yet)",
             fontsize=12, fontweight="bold")
ax.set_xticks(YEARS_MED)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("Observation: both groups declining before 2014 — the common trend.")
print("The gap that opens after 2014 for expansion states is the treatment effect.")

---
## 16) Write Your Prior — Medicaid Expansion

**Before computing any DiD estimate, answer all four questions.**  
Do this in the answer cell below — not after running the regressions.

---

**Q1.** From the summary table and raw plot above:  
Which group has a higher uninsured rate in 2012? In 2013?  
Does this surprise you given that expansion states chose to expand?  
*(Hint: think about the political economy of who chooses to expand.)*

**Q2.** If Medicaid expansion reduces uninsurance, what **sign** do you expect
for the DiD coefficient? Is this the same sign as in Card & Krueger, or opposite?

**Q3.** The uninsured rate was already falling in **both** groups before 2014
(the common trend of −0.8 pp/year).  
Suppose you compared expansion states' 2013 rate to their 2014 rate — just a simple
before–after comparison, ignoring the control group.  
Would that *over*- or *under*-estimate the treatment effect? By approximately how much?

**Q4 (harder).** In reality, many expansion states ran aggressive outreach campaigns,
enrolled people in precursor programs, and had progressive health policy environments
*even before 2014*.  
Does this make the parallel trends assumption more or less credible? Explain your reasoning
in two sentences.


**Your Prior (answer before running Section 17):**

**Q1 — Which group has higher uninsured rate, and does it surprise you?**

*TODO*

**Q2 — Expected sign of the DiD coefficient:**

*TODO*

**Q3 — Before–after comparison: over or underestimate?**

*TODO*

**Q4 — Does the political economy of expansion choice threaten parallel trends?**

*TODO*


---
## 17) The 2×2 DiD Table — Medicaid

We first use the **2013 vs. 2014** comparison — one pre-period, one post-period.
This is the cleanest version; it matches the Card & Krueger structure exactly.

Later in the event study (Section 20) we will use all four years.

Recall the DiD formula:
$$
\hat{\delta}^{DiD} = \bigl(\bar{Y}_{\text{Exp,2014}} - \bar{Y}_{\text{Exp,2013}}\bigr)
                  - \bigl(\bar{Y}_{\text{Non,2014}} - \bar{Y}_{\text{Non,2013}}\bigr)
$$


In [ ]:
# 17) 2×2 DiD Table — using 2013 (pre) and 2014 (post)

df_med_2x2 = df_med[df_med["year"].isin([2013, 2014])].copy()

means_2x2 = (
    df_med_2x2.groupby(["group_label", "year"])["uninsured_rate"]
              .mean()
              .round(3)
)

# ── Extract the four cells ─────────────────────────────────────────────────
exp_pre   = means_2x2["Expansion"][2013]
exp_post  = means_2x2["Expansion"][2014]
non_pre   = means_2x2["Non-Expansion"][2013]
non_post  = means_2x2["Non-Expansion"][2014]

delta_exp = exp_post - exp_pre     # Δ for expansion states (includes trend + effect)
delta_non = non_post - non_pre     # Δ for non-expansion (trend only)
did_med   = delta_exp - delta_non  # DiD removes the common trend

# ── Display as a formatted table ──────────────────────────────────────────
table_med = pd.DataFrame({
    "2013  (Pre)": [exp_pre, non_pre, "—"],
    "2014  (Post)": [exp_post, non_post, "—"],
    "Δ Post − Pre": [
        f"{delta_exp:+.3f} pp",
        f"{delta_non:+.3f} pp",
        f"{did_med:+.3f} pp  ← DiD"
    ]
}, index=["Expansion (treated)", "Non-Expansion (control)", "Difference (DiD)"])

print("=" * 60)
print("  2×2 DiD Table — Uninsured Rate (%), 2013 vs. 2014")
print("=" * 60)
display(table_med)

print()
print(f"  First difference — Expansion:      {delta_exp:+.3f} pp")
print(f"  First difference — Non-Expansion:  {delta_non:+.3f} pp")
print(f"  {'=' * 45}")
print(f"  DiD estimate:                      {did_med:+.3f} pp")
print(f"  True treatment effect:             {TRUE_EFFECT_MED:+.1f} pp")
print(f"  Difference (estimation error):     {did_med - TRUE_EFFECT_MED:+.3f} pp")
print(f"  {'=' * 45}")

print()
print("Interpretation:")
print(f"  Expansion states declined by {delta_exp:.2f} pp from 2013 to 2014.")
print(f"  Non-expansion states declined by {delta_non:.2f} pp over the same period")
print(f"  (this is the common trend we need to subtract).")
print(f"  Net of the common trend, expansion states declined an extra "
      f"{abs(did_med):.2f} pp — our DiD estimate.")

---
## 18) Visualize the DiD — 4 Years with Counterfactual

Now we use all four years to draw the DiD picture properly.

The **counterfactual line** answers: *what would expansion states' uninsured rate
have looked like after 2013 if they had followed the same path as non-expansion states?*

Construction:
- Anchor the counterfactual at the expansion group's 2013 mean.
- Apply the non-expansion group's year-over-year changes to project forward.
- The gap between the counterfactual and the actual expansion line is the DiD estimate.


In [ ]:
# 18) DiD Visualization — 4 years, counterfactual

gmeans = df_med.groupby(["group_label", "year"])["uninsured_rate"].mean()

exp_means = [gmeans["Expansion"][y]     for y in YEARS_MED]
non_means = [gmeans["Non-Expansion"][y] for y in YEARS_MED]

# ── Counterfactual: expansion states if they followed non-expansion's slope ──
# Anchored at expansion's 2013 value; apply non-expansion's changes year-by-year
base_idx   = YEARS_MED.index(2013)
counterfactual = [None] * len(YEARS_MED)
counterfactual[base_idx] = exp_means[base_idx]
for i in range(base_idx + 1, len(YEARS_MED)):
    counterfactual[i] = counterfactual[i-1] + (non_means[i] - non_means[i-1])
for i in range(base_idx - 1, -1, -1):
    counterfactual[i] = counterfactual[i+1] - (non_means[i+1] - non_means[i])

fig, ax = plt.subplots(figsize=(10, 6))

# ── Main lines ────────────────────────────────────────────────────────────
ax.plot(YEARS_MED, exp_means, color=TEAL,  marker="o", linewidth=2.5,
        markersize=9, label="Expansion states (treated)")
ax.plot(YEARS_MED, non_means, color=NAVY,  marker="s", linewidth=2.5,
        markersize=9, label="Non-expansion states (control)")
ax.plot(YEARS_MED, counterfactual, color=GOLD, linestyle="--",
        linewidth=2.2, marker="o", markersize=7,
        label="Expansion counterfactual (parallel trend)")

# ── Shade the gap (treatment effect) in post-period ───────────────────────
post_years  = [y for y in YEARS_MED if y >= TREAT_YEAR_MED]
post_idx    = [YEARS_MED.index(y) for y in post_years]

ax.fill_between(
    post_years,
    [exp_means[i]      for i in post_idx],
    [counterfactual[i] for i in post_idx],
    alpha=0.12, color=RED, label="Estimated treatment effect"
)

# ── Annotate DiD arrow at 2014 ─────────────────────────────────────────────
idx_2014 = YEARS_MED.index(2014)
y_actual = exp_means[idx_2014]
y_cf     = counterfactual[idx_2014]
did_viz  = y_actual - y_cf

ax.annotate("",
            xy=(2014.12, y_actual),
            xytext=(2014.12, y_cf),
            arrowprops=dict(arrowstyle="<->", color=RED, lw=2))
ax.text(2014.18, (y_actual + y_cf) / 2,
        f"DiD ≈ {did_viz:.1f} pp",
        color=RED, fontsize=10, va="center", fontweight="bold")

# ── Label data points ─────────────────────────────────────────────────────
for y, v in zip(YEARS_MED, exp_means):
    ax.text(y - 0.07, v - 0.35, f"{v:.1f}", ha="center",
            fontsize=9, color=TEAL)
for y, v in zip(YEARS_MED, non_means):
    ax.text(y + 0.07, v + 0.25, f"{v:.1f}", ha="center",
            fontsize=9, color=NAVY)

# ── Policy marker ─────────────────────────────────────────────────────────
ax.axvline(x=2013.5, color=GRAY, linestyle=":", linewidth=1.5, alpha=0.8)
ax.text(2013.55, ax.get_ylim()[0] + 0.4, "Expansion\nstarts →",
        color=GRAY, fontsize=9, va="bottom")

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Mean Uninsured Rate (%)", fontsize=12)
ax.set_title("Medicaid Expansion: DiD Visualization\n"
             "Uninsured Rate (%), 2012–2015",
             fontsize=13, fontweight="bold")
ax.set_xticks(YEARS_MED)
ax.legend(fontsize=10, loc="upper right")

plt.tight_layout()
plt.savefig(EXPORT_DIR / "medicaid_did_plot.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Expansion counterfactual at 2014: {y_cf:.3f}%")
print(f"Expansion actual at 2014:          {y_actual:.3f}%")
print(f"Visual DiD estimate:               {did_viz:.3f} pp")
print(f"True effect:                       {TRUE_EFFECT_MED:.1f} pp")

**Quick reflection after seeing the plot:**

1. Does the counterfactual slope look parallel to the non-expansion line in the pre-period? It should — we built the data that way. In the real world, you would need to *argue* for this.

2. The red-shaded region is the treatment effect. Notice it is present in **both** 2014 and 2015. What does this tell you about the durability of the Medicaid expansion effect?

3. In the plot, the two groups' pre-period lines are **not parallel** — they converge slightly. Is this a violation of parallel trends? *(Answer: no. The levels are different and the slopes are the same by construction. The convergence you see is just because the lines are declining at the same rate from different starting points — draw it on paper if unsure.)*


---
## 19) DiD as Regression — Medicaid

Same regression formula as Card & Krueger:

$$
\text{uninsured\_rate}_{st} =
    \beta_0
  + \beta_1 \cdot \text{expansion}_s
  + \beta_2 \cdot \text{post}_t
  + \beta_3 \cdot (\text{expansion}_s \times \text{post}_t)
  + \varepsilon_{st}
$$

| Coefficient | Meaning |
|:---|:---|
| $\beta_0$ | Baseline uninsured rate, non-expansion states, pre-period |
| $\beta_1$ | Level difference between expansion and non-expansion (pre-period) |
| $\beta_2$ | Common time trend — how much all states changed post-2014 |
| $\beta_3$ | **DiD estimate** — the causal effect of Medicaid expansion |

We use **HC3 robust standard errors** because state-level outcomes likely have
heteroskedastic residuals (larger, more diverse states have more variable outcomes).

In published work you would **cluster standard errors at the state level** —
we will discuss clustering when we cover panel fixed effects models later.


In [ ]:
# 19) DiD Regression — Medicaid Expansion (2013 vs 2014 subsample)
#
# We use the same 2×2 subsample as the table (2013 and 2014 only).
# In Section 20 we will use all four years.

if STATSMODELS:
    formula_med = "uninsured_rate ~ expansion + post + treated_x_post"
    model_med   = smf.ols(formula_med, data=df_med_2x2).fit(cov_type="HC3")

    print(model_med.summary())
    print()

    # ── Extract the key quantities ─────────────────────────────────────────
    beta0_med    = model_med.params["Intercept"]
    beta1_med    = model_med.params["expansion"]
    beta2_med    = model_med.params["post"]
    beta_did_med = model_med.params["treated_x_post"]
    pval_did_med = model_med.pvalues["treated_x_post"]
    ci_lo_med, ci_hi_med = model_med.conf_int().loc["treated_x_post"]

    print("=" * 60)
    print(f"  β₀ (non-expansion baseline)   : {beta0_med:+.3f}%")
    print(f"  β₁ (expansion level diff)     : {beta1_med:+.3f} pp")
    print(f"  β₂ (common time trend)        : {beta2_med:+.3f} pp")
    print(f"  β₃ (DiD — treatment effect)   : {beta_did_med:+.3f} pp  ← key estimate")
    print(f"  p-value (β₃)                  : {pval_did_med:.4f}")
    print(f"  95% CI (β₃)                   : [{ci_lo_med:.3f}, {ci_hi_med:.3f}]")
    print(f"  True treatment effect         : {TRUE_EFFECT_MED:+.1f} pp")
    print("=" * 60)

    print()
    print("Coefficient interpretation:")
    print(f"  β₁ = {beta1_med:.3f}: expansion states had an uninsured rate "
          f"{abs(beta1_med):.1f} pp {'lower' if beta1_med < 0 else 'higher'} "
          f"than non-expansion states before the policy.")
    print(f"  β₂ = {beta2_med:.3f}: both groups declined by {abs(beta2_med):.2f} pp "
          f"from 2013 to 2014 regardless of expansion status.")
    print(f"  β₃ = {beta_did_med:.3f}: on top of the common trend, "
          f"expansion states declined an extra {abs(beta_did_med):.2f} pp — "
          f"the estimated causal effect.")
else:
    print("statsmodels not available.")
    print(f"DiD estimate from 2×2 table: {did_med:+.3f} pp")
    beta_did_med = did_med
    pval_did_med = None

**Compare to the 2×2 table:**

The regression coefficient β₃ should equal the 2×2 DiD estimate from Section 17 exactly.
Verify this — if they differ, something went wrong.

**On β₂ — the common trend coefficient:**

β₂ captures the average decline in uninsured rates from 2013 to 2014 for non-expansion
states (the control group). In our simulation it should be close to `TIME_TREND_MED = -0.8`.
Check: is it? Why might it differ slightly from the calibrated value?


---
## 20) Event Study — 4 Periods

The **event study** is the standard tool for evaluating the parallel trends assumption.
It plots the DiD coefficient at each year relative to a **baseline year** (here, 2013 —
the last year before expansion).

**How to read an event study:**

| Period | Expected coefficient under parallel trends |
|:---|:---|
| Pre-treatment (2012, t = −2) | ≈ 0 (no treatment yet; groups moving in parallel) |
| Baseline (2013, t = −1) | = 0 by construction (the reference year) |
| Post-treatment (2014, t = 0) | Reflects the treatment effect |
| Post-treatment (2015, t = +1) | Reflects the (possibly dynamic) treatment effect |

If the pre-period coefficient is large and significantly different from zero —
the treated group was already diverging from the control group *before* the policy.
That is a red flag for the parallel trends assumption.

**Method used here:** manual group-means approach.
For each year $t$, compute:
$$
\hat{\beta}_t = 
  \bigl(\bar{Y}_{\text{Exp},t} - \bar{Y}_{\text{Exp},2013}\bigr)
- \bigl(\bar{Y}_{\text{Non},t} - \bar{Y}_{\text{Non},2013}\bigr)
$$

In published work, these coefficients come from a regression with year×treatment
interactions. The manual approach gives identical point estimates when there are
only two groups and balanced panels.


In [ ]:
# 20.1) Compute event study coefficients — clean data

BASE_YEAR_MED = 2013   # reference year (last pre-period)

gmeans_es = df_med.groupby(["group_label", "year"])["uninsured_rate"].mean()

event_coefs = []
for year in YEARS_MED:
    delta_exp_es = gmeans_es["Expansion"][year]     - gmeans_es["Expansion"][BASE_YEAR_MED]
    delta_non_es = gmeans_es["Non-Expansion"][year] - gmeans_es["Non-Expansion"][BASE_YEAR_MED]
    event_coefs.append({
        "year"    : year,
        "year_rel": year - TREAT_YEAR_MED,
        "coef"    : round(delta_exp_es - delta_non_es, 3)
    })

df_event_med = pd.DataFrame(event_coefs)

print("Event study coefficients relative to 2013 (clean data):")
print()
print(f"{'Year':<8} {'t':<8} {'Coef (pp)':<12} {'Interpretation'}")
print("-" * 55)
for _, row in df_event_med.iterrows():
    if row["year"] == BASE_YEAR_MED:
        interp = "← reference year (always 0 by construction)"
    elif row["year"] < TREAT_YEAR_MED:
        flag   = "✓ near zero" if abs(row["coef"]) < 0.5 else "✗ pre-trend!"
        interp = f"pre-treatment  {flag}"
    else:
        interp = "post-treatment (treatment effect)"
    print(f"{row['year']:<8} {row['year_rel']:+<8} {row['coef']:+<12.3f} {interp}")

In [ ]:
# 20.2) Plot the event study — clean data

fig, ax = plt.subplots(figsize=(9, 5))

colors_es = [NAVY if yr < TREAT_YEAR_MED else TEAL
             for yr in df_event_med["year"]]

bars = ax.bar(df_event_med["year_rel"], df_event_med["coef"],
              color=colors_es, alpha=0.85, edgecolor="white", width=0.5)

# Reference lines
ax.axhline(0,               color=GRAY, linewidth=1.5, linestyle="-")
ax.axhline(TRUE_EFFECT_MED, color=RED,  linewidth=1.5, linestyle="--",
           label=f"True effect = {TRUE_EFFECT_MED} pp")
ax.axvline(-0.5, color=GRAY, linewidth=1.2, linestyle=":", alpha=0.7)
ax.text(-0.45, ax.get_ylim()[0] * 0.88,
        "Expansion\nstarts →", color=GRAY, fontsize=9, ha="left")

# Annotate bar values
for _, row in df_event_med.iterrows():
    offset = 0.15 if row["coef"] >= 0 else -0.25
    ax.text(row["year_rel"], row["coef"] + offset,
            f"{row['coef']:+.2f}",
            ha="center", fontsize=11, fontweight="bold",
            color=NAVY if row["year"] < TREAT_YEAR_MED else TEAL)

# X-axis labels
ax.set_xticks(df_event_med["year_rel"])
ax.set_xticklabels(
    [f"{yr}\n(t={rel:+d})" for yr, rel in
     zip(df_event_med["year"], df_event_med["year_rel"])],
    fontsize=10
)

pre_patch  = mpatches.Patch(color=NAVY, alpha=0.85, label="Pre-treatment years")
post_patch = mpatches.Patch(color=TEAL, alpha=0.85, label="Post-treatment years")
ax.legend(handles=[pre_patch, post_patch,
                   plt.Line2D([0], [0], color=RED, linestyle="--",
                              label=f"True effect = {TRUE_EFFECT_MED} pp")],
          fontsize=10)

ax.set_xlabel("Year (t relative to expansion start)", fontsize=11)
ax.set_ylabel("pp relative to 2013 baseline", fontsize=11)
ax.set_title("Event Study — Medicaid Expansion\n"
             "Parallel Pre-Trend Test (Clean Case)",
             fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(EXPORT_DIR / "medicaid_event_study_clean.png", dpi=150, bbox_inches="tight")
plt.show()

pre_coef = df_event_med.loc[df_event_med["year"] == 2012, "coef"].values[0]
print(f"Pre-trend coefficient at t=−2 (2012): {pre_coef:+.3f} pp")
print("  → Should be near zero. If it is, parallel trends holds in the pre-period.")
print(f"\nPost-treatment coefficient at t=0 (2014): "
      f"{df_event_med.loc[df_event_med['year']==2014,'coef'].values[0]:+.3f} pp")
print(f"Post-treatment coefficient at t=+1 (2015): "
      f"{df_event_med.loc[df_event_med['year']==2015,'coef'].values[0]:+.3f} pp")

**What to look for:**

- The pre-treatment bar (t = −2, year 2012) should be close to zero.
  This is evidence that parallel trends held in the pre-period.

- The post-treatment bars (t = 0 and t = +1) should be near the true effect
  of −4.5 pp. Small deviations are just sampling noise.

- Having two pre-period observations (t = −2 and t = −1) is the advantage
  this design has over Card & Krueger's single-pre-period design.
  With only one pre-period, you cannot test parallel trends at all.


---
## 21) Break the Estimator — Pre-Existing Trend in Expansion States

**The real-world concern with Medicaid DiD studies:**

Expansion states were not chosen at random. They tended to be politically
progressive, wealthier, with stronger labor markets recovering faster from
the Great Recession — and with more aggressive pre-ACA health outreach programs.

All of this means expansion states may have been on a **steeper downward trend
in uninsurance even before 2014**, independent of the Medicaid expansion itself.

We simulate this with a `PRE_TREND_BIAS` parameter: an extra annual decline
applied only to expansion states in the pre-period.

**Your task:** observe how the event study changes, and how the DiD estimate
is biased as a result.

$$
\text{Bias} = \hat{\delta}^{DiD}_{\text{broken}} - \text{TRUE\_EFFECT\_MED}
$$

**Direction of bias:** if expansion states were already declining faster,
DiD attributes some of that pre-existing trend to the policy.
The estimate will be *more negative* than the true effect — we *overstate* the impact.


In [ ]:
# 21.1) Simulate the "broken" version — expansion states had a pre-existing trend

np.random.seed(99)   # same seed so state fixed effects are identical

PRE_TREND_BIAS = -0.9  # extra pp/year decline in expansion states BEFORE 2014 only
                        # This violates parallel trends by construction

rows_broken = []
for state_id in range(N_EXP + N_NON):
    is_exp   = (state_id < N_EXP)
    state_fe = np.random.normal(0, STATE_FE_SD)   # same draw as clean version
    baseline = BASELINE_EXP if is_exp else BASELINE_NON

    for year in YEARS_MED:
        t    = year - 2012
        post = int(year >= TREAT_YEAR_MED)

        # Extra pre-trend: only in expansion states, only before treatment
        pre_bias = PRE_TREND_BIAS * t * int(is_exp) * (1 - post)
        effect   = TRUE_EFFECT_MED if (is_exp and post) else 0.0
        noise    = np.random.normal(0, NOISE_SD_MED)

        unins = baseline + TIME_TREND_MED * t + pre_bias + effect + state_fe + noise

        rows_broken.append({
            "state_id"      : state_id,
            "year"          : year,
            "expansion"     : int(is_exp),
            "post"          : post,
            "year_rel"      : year - TREAT_YEAR_MED,
            "uninsured_rate": round(unins, 3),
            "group_label"   : "Expansion" if is_exp else "Non-Expansion",
            "treated_x_post": int(is_exp and post)
        })

df_med_broken = pd.DataFrame(rows_broken)

# Quick check — compute DiD from the broken data
df_med_broken_2x2 = df_med_broken[df_med_broken["year"].isin([2013, 2014])].copy()
gm_b = df_med_broken_2x2.groupby(["group_label", "year"])["uninsured_rate"].mean()

did_broken = ((gm_b["Expansion"][2014]     - gm_b["Expansion"][2013]) -
              (gm_b["Non-Expansion"][2014] - gm_b["Non-Expansion"][2013]))

print("=" * 50)
print(f"  True treatment effect : {TRUE_EFFECT_MED:+.1f} pp")
print(f"  DiD (clean data)      : {did_med:+.3f} pp")
print(f"  DiD (broken data)     : {did_broken:+.3f} pp")
print(f"  Bias = broken − true  : {did_broken - TRUE_EFFECT_MED:+.3f} pp")
print("=" * 50)
print()
print(f"The broken estimator overstates the negative effect by "
      f"{abs(did_broken - TRUE_EFFECT_MED):.2f} pp.")
print(f"The pre-trend bias ({PRE_TREND_BIAS:+.1f} pp/yr) accumulated over "
      f"2 pre-years = {2*PRE_TREND_BIAS:.1f} pp of total bias.")

In [ ]:
# 21.2) Side-by-side event studies — clean vs. broken

def compute_event_coefs(df, base_year, treat_year, years):
    """Compute manual event study coefficients relative to base_year."""
    gm = df.groupby(["group_label", "year"])["uninsured_rate"].mean()
    coefs = []
    for year in years:
        dt = gm["Expansion"][year]     - gm["Expansion"][base_year]
        dc = gm["Non-Expansion"][year] - gm["Non-Expansion"][base_year]
        coefs.append({"year": year, "year_rel": year - treat_year, "coef": dt - dc})
    return pd.DataFrame(coefs)


def plot_event_study_panel(ax, df, base_year, treat_year, years,
                           true_effect, title):
    """Plot a single event study panel."""
    df_es = compute_event_coefs(df, base_year, treat_year, years)

    colors_p = [NAVY if yr < treat_year else TEAL for yr in df_es["year"]]
    ax.bar(df_es["year_rel"], df_es["coef"],
           color=colors_p, alpha=0.85, edgecolor="white", width=0.5)

    ax.axhline(0,           color=GRAY, linewidth=1.2, linestyle="-")
    ax.axhline(true_effect, color=RED,  linewidth=1.8, linestyle="--",
               label=f"True effect = {true_effect} pp")
    ax.axvline(-0.5, color=GRAY, linewidth=1.2, linestyle=":", alpha=0.6)

    for _, row in df_es.iterrows():
        offset = 0.18 if row["coef"] >= 0 else -0.28
        ax.text(row["year_rel"], row["coef"] + offset, f"{row['coef']:+.2f}",
                ha="center", fontsize=11, fontweight="bold",
                color=NAVY if row["year"] < treat_year else TEAL)

    ax.set_xticks(df_es["year_rel"])
    ax.set_xticklabels(
        [f"{yr}\n(t={rel:+d})" for yr, rel in zip(df_es["year"], df_es["year_rel"])],
        fontsize=9
    )
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Year (t relative to expansion)")
    ax.set_ylabel("pp relative to 2013")
    ax.legend(fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

plot_event_study_panel(
    axes[0], df_med, BASE_YEAR_MED, TREAT_YEAR_MED, YEARS_MED,
    TRUE_EFFECT_MED,
    title="Clean: Parallel Pre-Trends ✓"
)

plot_event_study_panel(
    axes[1], df_med_broken, BASE_YEAR_MED, TREAT_YEAR_MED, YEARS_MED,
    TRUE_EFFECT_MED,
    title=f"Broken: Expansion States Already Trending Down ✗\n"
          f"(pre-trend bias = {PRE_TREND_BIAS} pp/yr)"
)

plt.suptitle("Event Study: When Parallel Trends Holds vs. Fails — Medicaid Expansion",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "medicaid_parallel_trends_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()

clean_coef_2012  = compute_event_coefs(df_med,        BASE_YEAR_MED,
                                       TREAT_YEAR_MED, YEARS_MED)
broken_coef_2012 = compute_event_coefs(df_med_broken, BASE_YEAR_MED,
                                       TREAT_YEAR_MED, YEARS_MED)

c0 = clean_coef_2012.loc[clean_coef_2012["year"] == 2012, "coef"].values[0]
b0 = broken_coef_2012.loc[broken_coef_2012["year"] == 2012, "coef"].values[0]

print(f"Pre-trend coefficient at t=−2 (2012):")
print(f"  Clean data:  {c0:+.3f} pp  ← near zero, parallel trends holds")
print(f"  Broken data: {b0:+.3f} pp  ← clearly negative, pre-trend detected!")
print()
print("The broken left panel shows a negative pre-period bar —")
print("a visible red flag that should make you distrust the post-period estimates.")

**Written response — after running the cells above:**

1. In the broken event study (right panel), what does the t = −2 coefficient tell you?
   Is the violation subtle or obvious? Would you catch this without the pre-period data?

2. The bias = broken DiD − true effect. Is the bias positive or negative?
   Explain *why* the direction of bias goes that way — in one sentence.

3. In published Medicaid studies, researchers try to address this concern by:
   (a) finding states that only recently became eligible ("late expanders" as controls),
   (b) matching expansion and non-expansion states on pre-expansion trends.
   Without re-running any code, explain intuitively why either of these approaches
   would improve the credibility of the parallel trends assumption.


**Your answers:**

1. *TODO*

2. *TODO*

3. *TODO*


---
## 22) Comparison: Card & Krueger vs. Medicaid Expansion

You have now run DiD in two very different settings.
Fill in the comparison table below, then answer the discussion questions.

| Feature | Card & Krueger (1994) | Medicaid Expansion (2014) |
|:---|:---|:---|
| Unit of analysis | Restaurant | State |
| Outcome | Employment (count) | Uninsured rate (%) |
| Outcome direction | ? with treatment | ? with treatment |
| Number of periods | 2 (1 pre, 1 post) | 4 (2 pre, 2 post) |
| Treated group baseline | Higher than control? | Higher than control? |
| Identifying variation | NJ vs. PA geography | State expansion choice |
| Selection concern | Geographic / state | Political economy |
| Can test parallel trends? | No (1 pre-period) | Yes (2 pre-periods) |

*(Fill in the "?" entries above before moving on.)*


**Discussion questions:**

**Q1 — Which design do you find more credible, and why?**  
Think about selection into treatment and the plausibility of parallel trends in each case.

*TODO*

**Q2 — How does having more pre-periods change what you can learn?**  
Specifically: what could you test in the Medicaid design that you *cannot* test in
Card & Krueger, and why does that matter?

*TODO*

**Q3 — Both studies use aggregate group-level data (state or restaurant) rather than
individual-level data. What are the tradeoffs of working at the aggregate level?**  
*(Think about statistical power, interpretation, and the kinds of mechanisms you can study.)*

*TODO*


---
## 23) Your Turn — Medicaid Expansion

Complete **at least two** of the three exercises below.

---

**Exercise A — What if there is no effect?**  
Change `TRUE_EFFECT_MED = 0.0` in Section 14 and re-run Sections 14 through 21.  

- What do the 2×2 table, the visualization, and the regression coefficient look like?
- Is the DiD estimate exactly zero? Why or why not?
- Does the regression p-value change? Explain what that tells you about the null hypothesis test.
- Does the event study look different from the clean version with `TRUE_EFFECT_MED = -4.5`?

Write a 3–4 sentence interpretation after re-running.


In [ ]:
# Exercise A — Your code here
# Tip: you can copy and modify the simulation cell from Section 14.
# Remember to use a different dataframe name so you don't overwrite df_med.


**Exercise A — Written interpretation:**

*TODO*


---

**Exercise B — Asymmetric common trends (a subtler violation)**  
Set `TRUE_EFFECT_MED = -4.5` (original) but change `TIME_TREND_MED` so that the
two groups have *different* common trends:

Modify the simulation so expansion states have a time trend of `−0.8 pp/yr` but
non-expansion states have a trend of `−0.3 pp/yr` (slower decline).

- Re-run the 2×2 table and the event study.
- Is the DiD biased? In which direction?
- Look at the pre-trend coefficient at t = −2. Does it flag the problem?

This scenario reflects a world where the two groups were on genuinely different
economic trajectories — a more realistic and harder-to-detect violation than
the one we simulated in Section 21.


In [ ]:
# Exercise B — Your code here
# Tip: you need to modify the simulation loop to apply different TIME_TRENDs
# for expansion vs. non-expansion states. Only one line in the loop changes.


**Exercise B — Written interpretation:**

*TODO*


---

**Exercise C — Evaluating a real-world Medicaid study claim**  
*(No coding required — this is a written critical analysis exercise.)*

A policy brief claims:

> *"We compared uninsured rates in expansion states in 2013 and 2015,
> and found a decline of 6.2 percentage points.
> This shows that Medicaid expansion reduced uninsurance by 6.2 pp."*

Using what you have learned in this notebook:

1. Identify **two specific methodological problems** with this claim.  
   Be precise — name the bias, explain the direction, and say how large it might be
   based on your simulation.

2. What would a proper DiD design need to do to isolate the causal effect?
   Name the control group, the comparison structure, and the assumption required.

3. **Harder:** Even a proper DiD estimate of −4.5 pp may not represent the
   effect on the "average American." Who specifically benefited from Medicaid expansion,
   and does your estimate apply to them, to everyone, or to some subset?
   *(This is asking you to think about external validity and local average treatment effects.)*


**Exercise C — Written answers:**

1. *TODO*

2. *TODO*

3. *TODO*


---
## 24) Export — Part II Files


In [ ]:
# 24) Export Part II datasets and print file manifest

df_med.to_csv(CLEAN_DIR / "medicaid_expansion_clean.csv", index=False)
df_med_broken.to_csv(CLEAN_DIR / "medicaid_expansion_broken.csv", index=False)
df_event_med.to_csv(CLEAN_DIR / "medicaid_event_study_coefs.csv", index=False)

print("=== Part II — Files written ===")
for f in sorted(list(CLEAN_DIR.glob("medicaid*")) + list(EXPORT_DIR.glob("medicaid*"))):
    print(f"  {f}")

print()
print("=== Part II — Final DiD Summary ===")
print(f"  True treatment effect         : {TRUE_EFFECT_MED:+.1f} pp")
print(f"  2×2 DiD estimate (2013→2014)  : {did_med:+.3f} pp")
if STATSMODELS:
    print(f"  Regression coeff β₃           : {beta_did_med:+.3f} pp")
    print(f"  p-value                       : {pval_did_med:.4f}")
    print(f"  95% CI                        : [{ci_lo_med:.3f}, {ci_hi_med:.3f}]")
print(f"  DiD (broken/biased data)      : {did_broken:+.3f} pp")
print(f"  Bias from pre-existing trend  : {did_broken - TRUE_EFFECT_MED:+.3f} pp")

print()
print("=== Notebook Complete ===")

---
## Part II — Key Takeaways

1. **DiD is a framework, not a recipe tied to one famous paper.**  
   Card & Krueger and the Medicaid expansion use identical math — they differ in
   what varies, at what level, and how credible the control group is.

2. **Levels can differ; trends must be parallel.**  
   Non-expansion states had higher baseline uninsured rates than expansion states.
   That did not matter — DiD subtracts out permanent group differences.
   What *does* matter is whether the two groups were on the same pre-treatment trend.

3. **More pre-periods = better test of parallel trends.**  
   With one pre-period (Card & Krueger), you cannot check whether the assumption holds.
   With two pre-periods (Medicaid), you can. This is why researchers collect
   historical data aggressively — not to use it directly, but to test the design.

4. **Event studies are the standard credibility check.**  
   Pre-period coefficients near zero = passing the test.  
   Large negative pre-period bar = the estimator was already biased before the policy happened.

5. **Selection into treatment drives the hardest identification problems.**  
   States chose whether to expand. That choice was correlated with health policy
   trajectories, political culture, and economic conditions — all potential confounders.
   There is no perfect fix. The DiD researcher's job is to make the strongest
   *defensible* case, not a perfect one.

---
**Next topic:** Text and Unstructured Data — collecting and analyzing data
that does not arrive pre-tabulated.
